In [2]:
import os
from dotenv import load_dotenv
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from pydantic import BaseModel
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException


load_dotenv()
os.environ["CEREBRAS_API_KEY"] = os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]     = os.getenv("GROQ_API_KEY")
BREVO_KEY = os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")
llm_gpt = ChatCerebras(
    model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"),
)
llm_groq = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
)


In [43]:
class User(BaseModel):
    id: int
    name: str
    email: str
    minutes_listened: int
    country: str
    joined_at: str


class EmailDraft(BaseModel):
    recipient: str
    subject: str
    body: str
    reason: str


class CampaignResult(BaseModel):
    total_users: int
    emails_sent: int
    failed: int


In [14]:
from pymongo import MongoClient

client = MongoClient(MONGODB_URI)

db = client["soulsync"]

users_collection = db["users"]

print(users_collection)


Collection(Database(MongoClient(host=['ac-jaaey03-shard-00-00.k7w05ba.mongodb.net:27017', 'ac-jaaey03-shard-00-01.k7w05ba.mongodb.net:27017', 'ac-jaaey03-shard-00-02.k7w05ba.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='SoulSync', authsource='admin', replicaset='atlas-99cwym-shard-0', tls=True), 'soulsync'), 'users')


In [15]:
print(users_collection.count_documents({}))

163


In [16]:
print(users_collection.find_one({}, {"_id": 0}))

{'googleId': '101531911541168397334', '__v': 130, 'createdAt': datetime.datetime(2026, 3, 4, 8, 44, 49, 307000), 'email': 'lokeshvijayraina@gmail.com', 'likedSongs': [{'songId': 'd6_r1LCo', 'title': 'God Mode Begins', 'artist': 'Vishnu Edavan, Sai Abhyankkar, Gana Muthu', 'albumArt': 'https://c.saavncdn.com/875/Karuppu-Original-Motion-Picture-Soundtrack-Tamil-2026-20260525175054-500x500.jpg', 'duration': 55, 'downloadUrl': [{'quality': '12kbps', 'url': 'https://aac.saavncdn.com/875/2f39ec8f8980a85cfe201713ca81bb93_12.mp4'}, {'quality': '48kbps', 'url': 'https://aac.saavncdn.com/875/2f39ec8f8980a85cfe201713ca81bb93_48.mp4'}, {'quality': '96kbps', 'url': 'https://aac.saavncdn.com/875/2f39ec8f8980a85cfe201713ca81bb93_96.mp4'}, {'quality': '160kbps', 'url': 'https://aac.saavncdn.com/875/2f39ec8f8980a85cfe201713ca81bb93_160.mp4'}, {'quality': '320kbps', 'url': 'https://aac.saavncdn.com/875/2f39ec8f8980a85cfe201713ca81bb93_320.mp4'}]}, {'songId': 'rSNk80el', 'title': 'Thenpandi Cheemayile', 

In [17]:
from langchain_core.tools import tool

@tool
def GetTopUsers(limit: int):
    """
    Return the top users ranked by listening time.
    """

    return list(
        users_collection.find(
            {},
            {
                "_id": 0,
                "name": 1,
                "email": 1,
                "totalListeningTime": 1,
            }
        )
        .sort("totalListeningTime", -1)
        .limit(limit)
    )

In [63]:
@tool
def sendMail(
    sender: str,
    receiver: str,
    subject: str,
    body: str,
) -> str:
    """Send an email using Brevo."""

    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key["api-key"] = BREVO_KEY

    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(
        sib_api_v3_sdk.ApiClient(configuration)
    )

    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        sender={"name": "SoulSync", "email": "lokeshvijayraina@gmail.com"},
        to=[{"email": receiver}],
        subject=subject,
        html_content=body,
    )

    try:
        api_response = api_instance.send_transac_email(send_smtp_email)
        print("Success! Message ID:", api_response.message_id)
        print(sender, receiver, subject)
        return f"Mail sent successfully to {receiver}"
    except ApiException as e:
        print(" Error:", e)
        return f"Failed to send email: {e}"


In [67]:
draft_llm = llm_gpt.with_structured_output(EmailDraft)
top_users = GetTopUsers.invoke(
    {
        "limit": 5
    }
)
drafts = []

for user in top_users:

    draft = draft_llm.invoke(f"""
    You are LOOKOUT.

    Write a professional appreciation email.

    Recipient: {user.name}
        Email: {user.email}

    Reason:
    The user is one of our top listeners this month with
    {user.minutes_listened} listening minutes.

    Write a warm thank-you email.
    """)

    drafts.append(draft)

In [68]:
drafts

[EmailDraft(recipient='itslokeshx@gmail.com', subject='Thank You for Being a Top Listener!', body='Dear Loki,\n\nI hope you’re doing well. I’m reaching out to extend our heartfelt gratitude for your incredible support this month. With an astounding 451,000 listening minutes, you’ve been one of our top listeners, and your dedication truly stands out.\n\nYour enthusiasm for our content not only inspires our team but also helps shape the future of our platform. It’s listeners like you who drive us to continuously improve and bring fresh, engaging experiences to our community.\n\nThank you for your continued loyalty and for being such an integral part of our journey. We look forward to bringing you even more content that you’ll love in the coming months.\n\nWarm regards,\n[Your Name]\n[Your Position]\nLOOKOUT Team', reason='The user is one of our top listeners this month with 451000 listening minutes.'),
 EmailDraft(recipient='Ethan', subject='Thank You for Being One of Our Top Listeners!'

In [57]:
for draft in drafts:
    print("=" * 60)
    print(f"Recipient : {draft.recipient}")
    print(f"Subject   : {draft.subject}")
    print(f"Reason    : {draft.reason}")
    print("-" * 60)
    print(draft.body)
    print("=" * 60)

Recipient : itslokeshx@gmail.com
Subject   : A Heartfelt Thank You from LOOKOUT
Reason    : Top listener of the month with 451,000 listening minutes
------------------------------------------------------------
Dear Loki,

We are thrilled to express our sincerest gratitude for being one of our top listeners this month with an impressive 451,000 listening minutes! Your dedication to our content is truly valued and appreciated.

Your loyalty and commitment inspire us to continue delivering high-quality experiences for our users. We are grateful for your engagement and trust in our platform.

Thank you for being an integral part of our LOOKOUT community. We look forward to continuing to provide you with the best content possible.

Warm regards,
LOOKOUT Team
Recipient : ethan@example.com
Subject   : A Big Thank You from Lookout!
Reason    : Appreciation for being one of our top listeners this month with 11,980 listening minutes.
------------------------------------------------------------
D

In [69]:
agent = create_agent(
    model=llm_gpt,
    tools=[GetTopUsers, sendMail],
    system_prompt="""
You are LOOKOUT.

You have two tools:

1. GetTopUsers(limit)  – returns the highest-ranked users.
2. sendMail(sender, receiver, subject, body)  – sends an email.

Workflow:
1. Find the top users.
2. Write a personalised thank-you email for each.
3. Send each email.
4. Repeat until every user has received one.
"""
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Reward our top 5 users.

Sender: lokeshvijayraina@gmail.com

Write a professional thank-you email for each user and send it.
"""
            )
        ]
    }
)


Success! Message ID: <202607051730.62862808934@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com itslokeshx@gmail.com Thank You for Being a Top Listener
Success! Message ID: <202607051730.58777046012@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com ethan@example.com Thank You for Being a Top Listener
Success! Message ID: <202607051730.24086269222@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com david@example.com Thank You for Being a Top Listener
Success! Message ID: <202607051730.35837956438@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com liam@example.com Thank You for Being a Top Listener
Success! Message ID: <202607051731.55185725366@smtp-relay.mailin.fr>
lokeshvijayraina@gmail.com sarah@example.com Thank You for Being a Top Listener
